In [0]:
# import json

# from pyspark.sql import functions as F
# from pyspark.sql.window import Window


# # 1. Create a repeatable source fixture

# booking = {
#     "booking_id": "B001",
#     "passenger_id": "P001",
#     "booking_status": "CONFIRMED",
#     "created_at": "2026-11-14T07:55:00Z",
# }

# def assignment_event(payload, version, effective_at, emitted_at):
#     segment_id = payload["booking_segment_id"]
#     return {
#         "schema_version": 1,
#         "source_system": "booking_system",
#         "event_id": f"EVT_{segment_id}_V{version}",
#         "entity_type": "booking_segment",
#         "entity_id": segment_id,
#         "entity_version": version,
#         "effective_at": effective_at,
#         "emitted_at": emitted_at,
#         "payload": payload,
#     }

# incoming_events = [
#     assignment_event(
#         {
#             "booking_segment_id": "BS002",
#             "booking_id": "B002",
#             "original_flight_leg_id": "FL100_20261115",
#             "assigned_flight_leg_id": "FL100_20261115",
#             "assignment_status": "ASSIGNED",
#         },
#         1,
#         "2026-11-14T08:15:00Z",
#         "2026-11-14T08:16:00Z",
#     ),
#     assignment_event(
#         {
#             "booking_segment_id": "BS003",
#             "booking_id": "B003",
#             "original_flight_leg_id": "FL100_20261115",
#             "assigned_flight_leg_id": "FL100_20261115",
#             "assignment_status": "ASSIGNED",
#         },
#         1,
#         "2026-11-14T08:30:00Z",
#         "2026-11-14T08:31:00Z",
#     ),
#     assignment_event(
#         {
#             "booking_segment_id": "BS001",
#             "booking_id": booking["booking_id"],
#             "original_flight_leg_id": "FL100_20261115",
#             "assigned_flight_leg_id": "FL100_20261115",
#             "assignment_status": "ASSIGNED"
#         },
#         1,
#         "2026-11-14T08:00:00Z",
#         "2026-11-14T08:01:00Z"

#     ),
#     assignment_event(
#         {
#             "booking_segment_id": "BS001",
#             "booking_id": booking["booking_id"],
#             "original_flight_leg_id": "FL100_20261115",
#             "assigned_flight_leg_id": "FL200_20261115",
#             "assignment_status": "ASSIGNED"
#         },
#         2,
#         "2026-11-14T10:00:00Z",
#         "2026-11-14T10:01:00Z"

#     )
    
# ]





In [0]:
import json

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# 1. Create a repeatable source fixture

booking = {
    "booking_id": "B001",
    "passenger_id": "P001",
    "booking_status": "CONFIRMED",
    "created_at": "2026-11-14T07:55:00Z",
}

booking_segment = {
    "booking_segment_id": "BS001",
    "booking_id": booking["booking_id"],
    "original_flight_leg_id": "FL100_20261115",
    "assigned_flight_leg_id": "FL100_20261115",
    "assignment_status": "ASSIGNED",
}

rebooked_segment = {
    **booking_segment,
    "assigned_flight_leg_id": "FL200_20261115",
}

def assignment_event(payload, version, effective_at, emitted_at):
    segment_id = payload["booking_segment_id"]
    return {
        "schema_version": 1,
        "source_system": "booking_system",
        "event_id": f"EVT_{segment_id}_V{version}",
        "entity_type": "booking_segment",
        "entity_id": segment_id,
        "entity_version": version,
        "effective_at": effective_at,
        "emitted_at": emitted_at,
        "payload": payload,
    }

initial_assignment_event = assignment_event(
    booking_segment, 1, "2026-11-14T08:00:00Z", "2026-11-14T08:01:00Z"
)
current_assignment_event = assignment_event(
    rebooked_segment, 2, "2026-11-14T10:00:00Z", "2026-11-14T10:01:00Z"
)

additional_assignment_events = [
    assignment_event(
        {
            "booking_segment_id": "BS002",
            "booking_id": "B002",
            "original_flight_leg_id": "FL100_20261115",
            "assigned_flight_leg_id": "FL100_20261115",
            "assignment_status": "ASSIGNED",
        },
        1,
        "2026-11-14T08:15:00Z",
        "2026-11-14T08:16:00Z",
    ),
    assignment_event(
        {
            "booking_segment_id": "BS003",
            "booking_id": "B003",
            "original_flight_leg_id": "FL100_20261115",
            "assigned_flight_leg_id": "FL100_20261115",
            "assignment_status": "ASSIGNED",
        },
        1,
        "2026-11-14T08:30:00Z",
        "2026-11-14T08:31:00Z",
    ),
]

incoming_events = [
    current_assignment_event,
    initial_assignment_event,
    current_assignment_event,
    *additional_assignment_events,
]

In [0]:
CATALOG = "airline_disruption"
SCHEMA = "dev"
namespace = f"`{CATALOG}`.`{SCHEMA}`"

bronze_table = f"{namespace}.v02_bronze_assignment_deliveries"
history_table = f"{namespace}.v02_silver_assignment_history"
current_table = f"{namespace}.v02_silver_assignment_current"

data = []
for i,event in enumerate(incoming_events,start =1):
    data.append((i,json.dumps(event)))

bronze = spark.createDataFrame(
    data,
    schema="delivery_no INT, raw_event STRING"
    )

bronze.write.format("delta").mode("overwrite").saveAsTable(bronze_table)

#print(data)

In [0]:
bronze_df = spark.table(bronze_table)
bronze_df.select("*").show(truncate = False)

In [0]:
event_schema = """
    schema_version INT,
    source_system STRING,
    event_id STRING,
    entity_type STRING,
    entity_id STRING,
    entity_version INT,
    effective_at STRING,
    emitted_at STRING,
    payload STRUCT<
        booking_segment_id: STRING,
        booking_id: STRING,
        original_flight_leg_id: STRING,
        assigned_flight_leg_id: STRING,
        assignment_status: STRING
    >
"""
entity_key = ["source_system", "entity_type", "entity_id"]
version_key = [*entity_key, "entity_version"]

events = (
    spark.table(bronze_table)
    .select(F.from_json("raw_event", event_schema).alias("event"))
    .select("event.*")
    .dropDuplicates()
)


def fail_if_conflict(conflicts, message):
    event_conflicts = conflicts.first()
    if event_conflicts is not None:
        raise ValueError(f"{message} : {event_conflicts.asDict()}")
 

# Check 1: Did the same source reuse an event ID for different data?
# We already removed exact copies, so if two rows still share an ID
# within the same source, something in those rows is different.
 
entity_id_conflict = (
    events
    .groupBy("source_system","event_id")
    .count()
    .where(F.col("count") > 1)
)

fail_if_conflict(entity_id_conflict, "Duplicate entity_id") 

# Check 2: Do two events disagree about the same version of a segment?
# Different event IDs are fine if the effective time and payload match.
# For example, version 2 can't assign BS001 to both FL200 and FL300.

entity_version_conflict =(
    events
    .groupBy(*version_key,"source_system")
    .agg(
        F.countDistinct(F.struct("effective_at", "payload")).alias("variants")
    )
    .where(F.col("variants") > 1)
    
)
fail_if_conflict(entity_version_conflict, "Conflicting entity version")

In [0]:
# Silver history keeps one row for each version of a segment.
# If two events describe the same version, the earlier checks made
# sure they agree. Sort by event ID so we pick the same row each run.

history_window = Window.partitionBy(*version_key).orderBy("event_id")

history = (
    events.withColumn("row_number", F.row_number().over(history_window))
    .where(F.col("row_number") == 1)
    .drop("row_number")
)

history.write.format("delta").mode("overwrite").saveAsTable(history_table)

In [0]:
# 5. Silver current state: keep the highest version per entity

current_window = Window.partitionBy(*entity_key).orderBy(
    F.col("entity_version").desc()
)

current = (
    spark.table(history_table)
    .withColumn("_rn", F.row_number().over(current_window))
    .where(F.col("_rn") == 1)
    .select(
        "source_system",
        "entity_id",
        "entity_version",
        "event_id",
        "effective_at",
        "payload.booking_id",
        "payload.original_flight_leg_id",
        "payload.assigned_flight_leg_id",
        "payload.assignment_status",
    )
)

current.write.format("delta").mode("overwrite").saveAsTable(current_table)

In [0]:
# 6. Verify the milestone

print("Bronze deliveries:", spark.table(bronze_table).count())
print("Silver history rows:", spark.table(history_table).count())
print("Silver current rows:", spark.table(current_table).count())

In [0]:
spark.table(history_table).select("*").show(truncate = False)


In [0]:
from pyspark.sql import functions as F 
from pyspark.sql.window import Window

win = Window.partitionBy(*entity_key).orderBy("entity_version")


assignment_intervals = (
    spark.table(history_table)
    .withColumn("Valid_from", F.to_timestamp("effective_at"))
    .withColumn("Valid_to", F.lead("Valid_from", 1).over(win))
    .select(
        "entity_id",
        "entity_version",
        "payload.assigned_flight_leg_id",
        "Valid_from",
        "Valid_to",
    )
)

assignment_intervals.show()

In [0]:
cancellations = spark.createDataFrame(
    [
        ("C001", "FL100_20261115", "2026-11-14T09:00:00Z")
    ],
    "disruption_id STRING, flight_leg_id STRING, cancelled_at STRING",
).withColumn("cancelled_at", F.to_timestamp("cancelled_at"))



h = assignment_intervals.alias("h")
c = cancellations.alias("c")

affected_segments = (
    h.join(
        c,
        (F.col("h.assigned_flight_leg_id") == F.col("c.flight_leg_id"))
        & (F.col("c.cancelled_at") >= F.col("h.Valid_from"))
        & (F.col("h.Valid_to").isNull() | (F.col("c.cancelled_at") < F.col("h.Valid_to"))),
        "inner",
    )
    .select(
        "c.disruption_id",
        "c.flight_leg_id",
        "c.cancelled_at",
        "h.entity_id",
        "h.entity_version",
        "h.Valid_from",
        "h.Valid_to",
    ).distinct()
)

affected_segments.orderBy("disruption_id").show(truncate=False)

In [0]:
s = affected_segments.alias("s")
a = assignment_intervals.alias("a")

rebooked = (
    s.join(
        a,
        (F.col("s.entity_id") == F.col("a.entity_id"))
        & (F.col("a.entity_version") > F.col("s.entity_version"))
        & (F.col("a.Valid_from") > F.col("s.cancelled_at"))
        & (F.col("a.assigned_flight_leg_id") != F.col("s.flight_leg_id")),
        "left",
    )
    .select(
        "s.disruption_id",
        "s.entity_id",
        F.col("s.flight_leg_id").alias("old_flight_leg_id"),
        F.col("a.assigned_flight_leg_id").alias("new_flight_leg_id"),
        F.col("a.Valid_from").alias("rebooked_at"),
        F.when(F.col("a.entity_id").isNotNull(), "rebooked")
         .otherwise("unresolved").alias("status"),
    )
    .orderBy("s.disruption_id", "s.entity_id")
)

rebooked.show(truncate=False)

In [0]:
refund_requested_event = {
    "schema_version": 1,
    "source_system": "payment_system",
    "event_id": "EVT_RF002_V1",
    "entity_type": "refund",
    "entity_id": "RF002",
    "entity_version": 1,
    "effective_at": "2026-11-14T09:20:00Z",
    "emitted_at": "2026-11-14T09:21:00Z",
    "payload": {
        "refund_id": "RF002",
        "payment_id": "PAY002",
        "booking_segment_id": "BS002",
        "disruption_id": "C001",
        "amount": "5000.00",
        "currency": "INR",
        "status": "REQUESTED",
    },
}

refund_completed_event = {
    **refund_requested_event,
    "event_id": "EVT_RF002_V2",
    "entity_version": 2,
    "effective_at": "2026-11-14T11:00:00Z",
    "emitted_at": "2026-11-14T11:01:00Z",
    "payload": {
        **refund_requested_event["payload"],
        "status": "COMPLETED",
    },
}

refund_events = [
    refund_requested_event,
    refund_completed_event,
    refund_completed_event,  # Retry: same event ID and contents.
]

# refund_completed_event = spark.createDataFrame(
#     [refund_completed_event],
#     """
#     schema_version INT,
#     source_system STRING,
#     event_id STRING,
#     entity_type STRING,
#     entity_id STRING,
#     entity_version INT,
#     effective_at STRING,
#     emitted_at STRING,
#     payload STRUCT<
#         refund_id: STRING,
#         payment_id: STRING,
#         booking_segment_id: STRING,
#         disruption_id: STRING,
#         amount: STRING,
#         currency: STRING,
#         refund_status: STRING
#     >
#     """,
# )

#print(json.dumps(refund_completed_event, indent=2))
#refund_completed_event.show(truncate=False)

In [0]:



# total = 1500
# rc  = refund_completed_event.alias("rc")
# rb = rebooked.alias("rb")
# refunded = rc.join(
#     rb,F.col("rc.payload.booking_segment_id") == F.col("rb.entity_id"), 
#     "inner"
# ).select(
#     F.col("rb.entity_id"),
#     F.col("rc.payload.refund_status"),
#     F.col("rc.payload.amount"),
#     F.col("rc.payload.currency")
#     .when(F.col("rc.payload.refund_status") == "COMPLETED",total - F.col("rc.payload.amount")).
# )

# refunded.show(truncate=False)

In [0]:
#question: How much money has been paid to customers?
refund_schema = """
    schema_version INT,
    source_system STRING,
    event_id STRING,
    entity_type STRING,
    entity_id STRING,
    entity_version INT,
    effective_at STRING,
    emitted_at STRING,
    payload STRUCT<
        refund_id: STRING,
        payment_id: STRING,
        booking_segment_id: STRING,
        disruption_id: STRING,
        amount: STRING,
        currency: STRING,
        status: STRING
    >
"""

def refund_event(event_id, version, status, at):
    return {
        "schema_version": 1,
        "source_system": "payment_system",
        "event_id": event_id,
        "entity_type": "refund",
        "entity_id": "REF001",
        "entity_version": version,
        "effective_at": at,
        "emitted_at": at,
        "payload": {
            "refund_id": "REF001",
            "payment_id": "PAY002",
            "booking_segment_id": "BS002",
            "disruption_id": "C001",
            "amount": "5000.00",
            "currency": "INR",
            "status": status,
        },
    }

request = refund_event("EVT_REF001_V1", 1, "REQUESTED", "2026-11-14T09:05:00Z")
completion = refund_event("EVT_REF001_V2", 2, "COMPLETED", "2026-11-14T09:30:00Z")

def refunded_total(events):
    df = (
        spark.createDataFrame(
            [(json.dumps(e),) for e in events], "raw_event STRING"
        )
        .select(F.from_json("raw_event",refund_schema).alias("e"))
        .select("e.*")
        .dropDuplicates(["source_system", "event_id"])
        .withColumn("amount", F.col("payload.amount").cast("decimal(18,2)"))
        .withColumn("status", F.col("payload.status"))
    )
    w = Window.partitionBy("entity_id").orderBy(F.col("entity_version").desc())
    latest = df.withColumn("rn", F.row_number().over(w)).where("rn = 1")
    total = (
        latest.where("status = 'COMPLETED'")
        .agg(F.coalesce(F.sum("amount"), F.lit(0).cast("decimal(18,2)")).alias("t"))
        .first()["t"]
    )
    return total

import json
from decimal import Decimal

assert refunded_total([request]) == Decimal("0.00")
assert refunded_total([request, completion]) == Decimal("5000.00")
assert refunded_total([request, completion, completion]) == Decimal("5000.00")


print(refunded_total(refund_events))